
# Bigfoot Sightings Explorer EDA
In this section, I load the Bigfoot sightings dataset and explore its basic structure before creating visualizations(Streamlit).


## 1. Import Libraries

In [1]:
import pandas as pd
import altair as alt

## 2. Load the Bigfoot sightings dataset

In [2]:
#Bigfoot sightings dataset
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"
df = pd.read_csv(url)

df.head()

,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


### 2-1. Inspect the dataset structure

In [3]:
# Check the number of rows and columns
print("*Dataset shape:", df.shape)

# Display all column names
print("\nColumn names:")
print(df.columns.tolist())

# Check data types and non-null counts
df.info()

*Dataset shape: (4747, 29)

Column names:
['observed', 'location_details', 'county', 'state', 'season', 'title', 'latitude', 'longitude', 'date', 'number', 'classification', 'geohash', 'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point', 'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity', 'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index', 'visibility', 'wind_bearing', 'wind_speed', 'location']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4747 entries, 0 to 4746
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   observed            4711 non-null   object 
 1   location_details    4008 non-null   object 
 2   county              4747 non-null   object 
 3   state               4747 non-null   object 
 4   season              4747 non-null   object 
 5   title               3797 non-null   object 
 6   latitude            3797 non-null   float64
 7   lo

### 2-2. Check missing values 

In [4]:
# Count missing values in each column
missing_values = df.isnull().sum()

# Display columns with missing values
missing_values[missing_values > 0].sort_values(ascending=False)

precip_type           3187
pressure              2336
precip_probability    2254
precip_intensity      2254
visibility            1930
cloud_cover           1894
temperature_mid       1797
temperature_low       1794
temperature_high      1649
dew_point             1614
humidity              1614
wind_bearing          1600
wind_speed            1598
moon_phase            1591
uv_index              1591
summary               1591
title                  950
longitude              950
geohash                950
date                   950
latitude               950
location               950
location_details       739
observed                36
dtype: int64

In [5]:
# Check the distribution of sighting classifications
df["classification"].value_counts()

classification
Class B    2370
Class A    2347
Class C      30
Name: count, dtype: int64


## 3. Data Cleaning and Preparation

In this section, I convert the date column into datetime format, extract the year, and count the number of reported sightings for each year.

In [6]:
# Make a copy of the dataset
df_clean = df.copy()

# Convert date to datetime
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")       #Referred to: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html

# Get the year from the date
df_clean["year"] = df_clean["date"].dt.year

# Check the year range
print(df_clean["year"].min())
print(df_clean["year"].max())

1869.0
2018.0


In [7]:
# Count the number of sightings by year
yearly_counts = df_clean.groupby("year").size().reset_index(name="count")

# Preview the results
yearly_counts.head()

,year,count
0,1869.0,1
1,1921.0,1
2,1925.0,1
3,1930.0,1
4,1937.0,1


## 4. Visualization 1: Bigfoot Sightings Over Time

This visualization shows how the number of reported Bigfoot sightings has changed over time.

In [8]:
# Create a line chart of yearly sightings
chart1 = alt.Chart(yearly_counts).mark_line(strokeWidth=4).encode(                     # Make the line slightly thicker for easier hovering
    x=alt.X("year:Q", title="Year"),                                                   #Referred to: https://altair-viz.github.io/user_guide/marks/line.html 
    y=alt.Y("count:Q", title="Number of Sightings"),
    tooltip=["year:Q", "count:Q"],
).properties(
    title="Bigfoot Sightings Over Time",
    width=700,
    height=350
)

chart1

alt.Chart(...)

### 4-1. Adding Interactivity

I added a **slider** to let users choose the starting year and explore how the number of reported sightings changes over time.

In [9]:
# Create a slider to select the starting year
year_slider = alt.binding_range(                                #Referred to: https://altair-viz.github.io/altair-viz-v4/user_guide/interactions.html
    min=1869,
    max=2018,
    step=1,
    name="Start Year: "
)

# Set the initial year
year_param = alt.param(value=1869, bind=year_slider)            #Referred to: https://altair-viz.github.io/gallery/slider_cutoff.html

# Filter the chart based on the selected year                   #Referred to: https://altair-viz.github.io/user_guide/transform/filter.html
chart1_interactive = chart1.add_params(
    year_param
).transform_filter(
    alt.datum.year >= year_param
)

chart1_interactive

alt.Chart(...)

This line chart shows how the number of reported Bigfoot sightings has changed over time. I used a **line chart** because it makes it easier to see trends across years. The x-axis represents the **year**, while the y-axis shows the **number of reports**. 

I also made the line slightly thicker to improve visibility and make hovering easier. I added a slider so users can explore the data from different starting years. If I had more time, I would like to compare the trends across different states to see whether they follow similar patterns.


## 5. Visualization 2: Bigfoot Sightings by State

For the second visualization, I want to compare the number of reported Bigfoot sightings across different states.

In [10]:
# Count the number of sightings in each state
state_counts = df_clean.groupby("state").size().reset_index(name="count")

# Sort states by the number of sightings
state_counts = state_counts.sort_values("count", ascending=False)

# Check the top 10 states
state_counts.head(10)

,state,count
45,Washington,563
4,California,402
8,Florida,292
33,Ohio,276
35,Oregon,241
11,Illinois,232
41,Texas,215
20,Michigan,208
23,Missouri,141
9,Georgia,121


In [11]:
# Select the top 10 states
top_states = state_counts.head(10)

# Create a horizontal bar chart
chart2 = alt.Chart(top_states).mark_bar().encode(                  #Referred to: https://altair-viz.github.io/user_guide/marks/bar.html
    x=alt.X("count:Q", title="Number of Sightings"),
    y=alt.Y("state:N", sort="-x", title="State"),
    tooltip=["state:N", "count:Q"]
).properties(
    title="Top 10 States by Bigfoot Sightings",
    width=650,
    height=350
)

chart2

alt.Chart(...)

### 5-1. Adding Interactivity

I added a **slider** to let users choose how many states to display. This makes it easier to compare the states with the highest number of reported sightings.


In [12]:
# Create a slider to select the number of states
top_n_slider = alt.binding_range(
    min=5,
    max=20,
    step=1,
    name="Number of States: "
)

# Set the initial number of states
top_n = alt.param(value=10, bind=top_n_slider)

# Rank states by the number of sightings
chart2_interactive = alt.Chart(state_counts).transform_window(                    #Referred to: https://altair-viz.github.io/user_guide/transform/window.html & https://altair-viz.github.io/user_guide/generated/core/altair.WindowTransform.html
    rank="row_number()",
    sort=[alt.SortField("count", order="descending")]
).transform_filter(
    alt.datum.rank <= top_n
).mark_bar().encode(
    x=alt.X("count:Q", title="Number of Sightings"),
    y=alt.Y("state:N", sort="-x", title="State"),
    tooltip=["state:N", "count:Q"]
).add_params(
    top_n
).properties(
    title="Top States by Bigfoot Sightings",
    width=650,
    height=450
)

chart2_interactive

alt.Chart(...)

This horizontal bar chart compares the number of reported Bigfoot sightings across different states. Washington has the highest number of reports (563), followed by California (402) and Florida (292). 

I used a **horizontal bar chart** because it makes it easier to compare the counts across states and read their names. The bars are sorted in descending order to highlight the states with the most reports. I also added a **slider** so users can choose how many states to display, from 5 to 20. 

If I had more time, I would like to compare the number of sightings relative to each state's population, since states with larger populations may naturally have more reports.